In [1]:
%cd drive/MyDrive/Thesis/Models/PSSM_NN

/content/drive/MyDrive/Thesis/Models/PSSM_NN


In [2]:
!ls

inference.ipynb  model.py  __pycache__	saved_model  training.ipynb


In [3]:
import numpy as np
import torch
import torch.nn as nn

from model import CNN_BRNN

In [4]:
#----- training loop
def training_loop(epochs, dataloader, model, loss_func, opt):

  # iterations = len(train_dataset)//batch_size #len(dataset)//batch_size
  model.train()

  for epoch in range(epochs):
    print(f"Epoch[{epoch+1}/{epochs}]\n")
    
    acc_num = 0
    acc_den = 0

    for x,y in dataloader:      
      #forward
      x = x.to(device)
      y = y.to(device)
      outputs    = model(x)

      # print(outputs.shape)
      # output : (N x C x L)
      _, preds   = torch.max(outputs, 1)

      # print(preds.shape)
      # print(y.shape)
      acc_num += (preds == y).sum().item() 
      acc_den += len(y[y > -1])

      # y = y.view(-1)
      loss = loss_func(outputs, y)
      # cum_loss += loss.item()

      #backward
      opt.zero_grad() 
      loss.backward()

      #adam step
      opt.step()

      #scheduler step

    print(f'Accuracy: {(acc_num/acc_den):.3f}\n')
    # for x,y in validation_loader:


In [ ]:
class CNN_BRNN(nn.Module):
    def __init__(self, n_in=21, n_state=3, conv_hidden_size=64, rnn_hidden_size=256):
        super().__init__()

        # Conv1d(in_channels, out_channels, kernel_size, stride, padding)

        self.conv1 = nn.Sequential(nn.Conv1d( n_in, conv_hidden_size, 3, 1, 3 // 2), nn.ReLU())
        self.conv2 = nn.Sequential(nn.Conv1d( n_in, conv_hidden_size, 7, 1, 7 // 2), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Conv1d( n_in, conv_hidden_size, 11, 1, 11 // 2), nn.ReLU())

        # nn.GRU(n_input, n_hidden, n_layers, bias, batch_first, dropout, bidirectional)
        # bias – If False, then the layer does not use bias weights b_ih and b_hh. Default: True
        # batch_first – If True, then the input and output tensors are provided as (batch, seq, feature) instead of (seq, batch, feature).
        #               Note that this does not apply to hidden or cell states.
        # dropout – If non-zero, introduces a Dropout layer on the outputs of each GRU layer except the last layer, with dropout probability equal to dropout. Default: 0
        # bidirectional – If True, becomes a bidirectional GRU. Default: False

        self.brnn = nn.GRU(conv_hidden_size*3, rnn_hidden_size, 3, True, True, 0.5, True)

        self.fc = nn.Sequential(
                nn.Linear(rnn_hidden_size*2+conv_hidden_size*3, 128),
                nn.ReLU(),
                nn.Linear(128, n_state))

    def forward(self, x):
        # (batch_size , hidden_size , seq_len)
        conv_out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x)], dim=1)
        # (batch_size , seq_len , hidden_size)
        conv_out = conv_out.transpose(1, 2)

        # bidirectional rnn
        out, _ = self.brnn(conv_out)

        out = torch.cat([conv_out, out], dim=2)
        # print(out.sum())

        out = self.fc(out)
        #  (N x L x C) --> (N x C x L)
        out = out.transpose(1,2)
        return out


In [5]:
def transform_PSSM(PSSM):
    if PSSM.shape[-1] >= 700:
        return (PSSM[:,:700])
    else:
        return(torch.nn.functional.pad(PSSM, (700 - PSSM.shape[1],0)))

def transform_label(label_t):
    if label_t.shape[-1] >= 700:
        return(label_t[:700])
    else:
        return(torch.nn.functional.pad(label_t, (700 - label_t.shape[-1], 0), value = -9  ))

class np_dataset(torch.utils.data.Dataset):
    def __init__(self, np_data):
        self.np_data = np_data
        self.samples = np_data.files
        self.transform_PSSM = transform_PSSM
        self.transform_label = transform_label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        PSSM  = self.np_data[self.samples[idx]][3:]
        mask  = self.np_data[self.samples[idx]][1]
        label = self.np_data[self.samples[idx]][2]
        
        PSSM  = torch.from_numpy(PSSM)
        PSSM  = PSSM.to(torch.float)
        PSSM  = self.transform_PSSM(PSSM)
        
        label[mask==0] = -9
        label = torch.from_numpy(label).long()
        label = self.transform_label(label)
        return PSSM, label


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

np_data = np.load('../../Data/PSSM_train_data.npz')
# 0 -> primary, 1 -> Mask, 2 -> SS target, 3-24 -> PSSM

my_dataset = np_dataset(np_data)
dataloader = torch.utils.data.DataLoader(my_dataset, batch_size = 100, shuffle=True)
loss_func = torch.nn.CrossEntropyLoss(ignore_index = -9)
model = CNN_BRNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay = 0.01)


In [8]:
training_loop(epochs=10, dataloader=dataloader, model=model, loss_func=loss_func, opt=optimizer)

Epoch[1/10]

Accuracy: 0.755

Epoch[2/10]

Accuracy: 0.756

Epoch[3/10]

Accuracy: 0.758

Epoch[4/10]

Accuracy: 0.758

Epoch[5/10]

Accuracy: 0.759

Epoch[6/10]

Accuracy: 0.759

Epoch[7/10]

Accuracy: 0.760

Epoch[8/10]

Accuracy: 0.761

Epoch[9/10]

Accuracy: 0.761

Epoch[10/10]

Accuracy: 0.761



In [9]:
torch.save(model.state_dict(), './saved_model')